In [ ]:
import sys
print("🔧 Instalando dependencias: pydantic")
!{sys.executable} -m pip install -q pydantic
print("✅ Instalación completada. Ejecuta las celdas siguientes.")

: 

# AEM4L1 E01 — Pydantic: Modelo Básico

**Objetivo:** Entender qué es Pydantic, crear modelos simples con BaseModel y validación automática.

**Tipo:** Resuelto

**Uso:** listo para Google Colab. Ejecutá las celdas en orden.

## 📦 Dependencias

**Necesario instalar:**
- `pydantic`

**Cómo instalar:**
```bash
pip install pydantic
```

La celda de instalación está incluida a continuación. Ejecuta esa celda primero.

## ¿Qué es Pydantic?

Pydantic es una librería Python que:
- Define estructuras de datos con tipos (como en TypeScript)
- **Valida automáticamente** los datos al crear objetos
- Convierte ("parses") datos JSON ↔ objetos Python
- Genera errores útiles cuando algo no encaja

En nuestro caso, actuará como **barrera entre el LLM y tu aplicación**, asegurando que los datos del modelo sean correctos.

In [ ]:
import sys

print("=" * 60)
print("🔧 Instalando dependencias")
print("=" * 60)

# Instalar pydantic
print("\n📦 Instalando: pydantic")
!{sys.executable} -m pip install -q pydantic

print("✅ ¡Listo! Todas las dependencias instaladas.\n")

: 

## Importaciones necesarias

En este notebook vamos a usar:
- **pydantic.BaseModel**: Clase base para definir modelos de datos
- **pydantic.ValidationError**: Excepción para manejar errores de validación
- **json**: Para convertir entre JSON y diccionarios Python
- **typing.Optional**: Para campos opcionales (que pueden ser None)

## 1. Crear un modelo básico

In [ ]:
from pydantic import BaseModel

# MODELO BÁSICO: Clase que define una estructura de datos
# Cada atributo tiene un TYPE HINT (tipo de datos esperado)
class Persona(BaseModel):
    """
    Modelo para representar datos de una persona.
    Pydantic validará automáticamente que:
    - nombre sea un string
    - edad sea un entero
    - activo sea un booleano
    """
    nombre: str          # Debe ser texto
    edad: int            # Debe ser número entero
    activo: bool         # Debe ser True o False

print("=" * 60)
print("EJEMPLO 1: Crear una instancia CON DATOS CORRECTOS")
print("=" * 60)

# Crear una instancia (objeto) de la clase Persona
persona1 = Persona(nombre="Juan", edad=30, activo=True)
print(f"\n✅ Persona creada exitosamente:")
print(f"   Representación: {persona1}")
print(f"\nAcceso a campos:")
print(f"   - Nombre: {persona1.nombre}")
print(f"   - Edad: {persona1.edad}")
print(f"   - Activo: {persona1.activo}")

## 2. Validación automática: ¿qué pasa si los tipos son incorrectos?

In [ ]:
from pydantic import ValidationError

print("\n" + "=" * 60)
print("EJEMPLO 2: Validación AUTOMÁTICA - Datos INCORRECTOS")
print("=" * 60)

# Diccionario con tipos incorrectos
datos_incorrectos = {
    "nombre": "Maria",
    "edad": "treinta",  # ❌ ERROR: debe ser int, no str
    "activo": True
}

print("\nIntentando crear Persona con:")
print(f"  nombre='Maria' (str) ✅")
print(f"  edad='treinta' (str) ❌ [esperado: int]")
print(f"  activo=True (bool) ✅")

try:
    persona2 = Persona(**datos_incorrectos)
except ValidationError as e:
    print(f"\n❌ Pydantic detectó el error y lanzó ValidationError:")
    print(f"\nDetalles del error:")
    for error in e.errors():
        print(f"  - Campo: {error['loc'][0]}")
        print(f"  - Tipo esperado: {error['type']}")
        print(f"  - Mensaje: {error['msg']}")

## 3. Convertir datos JSON a objetos Python

In [ ]:
import json

print("\n" + "=" * 60)
print("EJEMPLO 3: PARSING - JSON → Objeto Pydantic")
print("=" * 60)

# Simular JSON que viene de un LLM o una API
json_string = '{"nombre": "Carlos", "edad": 25, "activo": false}'

print(f"\n📥 JSON recibido:")
print(f"   {json_string}")

# Paso 1: Convertir JSON (string) → Diccionario Python
datos = json.loads(json_string)
print(f"\n📦 Convertido a diccionario:")
print(f"   {datos}")

# Paso 2: Validar y convertir Diccionario → Objeto Pydantic
persona3 = Persona(**datos)

print(f"\n✅ Objeto Pydantic creado y VALIDADO:")
print(f"   {persona3}")
print(f"\nTipo de dato: {type(persona3)}")
print(f"¿Está activo? {persona3.activo}")
print(f"\n💡 Ahora podemos usar los datos como objeto Python normal")

## 4. Serializar objetos Pydantic de vuelta a JSON

In [ ]:
print("\n" + "=" * 60)
print("EJEMPLO 4: SERIALIZACIÓN - Objeto Pydantic → JSON")
print("=" * 60)

# Convertir objeto Pydantic → Diccionario Python
diccionario = persona1.model_dump()
print(f"\n📦 Objeto convertido a diccionario:")
print(f"   {diccionario}")

# Convertir Diccionario → JSON (string)
json_output = json.dumps(diccionario, indent=2)
print(f"\n📤 Convertido a JSON (para enviar a una API):")
print(json_output)

print(f"\n💡 Esto es útil para:")
print(f"   - Guardar datos en bases de datos")
print(f"   - Enviar datos a APIs")
print(f"   - Loguear datos en formatos estándar")

## 5. Campos opcionales

In [ ]:
from typing import Optional

print("\n" + "=" * 60)
print("EJEMPLO 5: Campos OPCIONALES")
print("=" * 60)

class PersonaExtendida(BaseModel):
    """
    Modelo con campos opcionales.
    - nombre: OBLIGATORIO
    - edad: OBLIGATORIO
    - email: OPCIONAL (puede ser None o un string)
    """
    nombre: str
    edad: int
    email: Optional[str] = None  # OPCIONAL, valor por defecto None

print("\n1️⃣ Crear sin el campo opcional (email):")
persona4 = PersonaExtendida(nombre="Ana", edad=28)
print(f"   ✅ {persona4}")
print(f"   Email es: {persona4.email}")

print("\n2️⃣ Crear CON el campo opcional (email):")
persona5 = PersonaExtendida(nombre="Luis", edad=35, email="luis@example.com")
print(f"   ✅ {persona5}")
print(f"   Email es: {persona5.email}")

print("\n💡 Si intento sin campos obligatorios, Pydantic error:")
try:
    persona6 = PersonaExtendida(nombre="Sofia")  # Falta 'edad'
except ValidationError as e:
    print(f"   ❌ {e.errors()[0]['msg']}")

## 6. Caso de uso: Extracción de datos de una imagen

print("\n" + "=" * 60)
print("CASO DE USO REAL: Extracción de datos de documentos")
print("=" * 60)

# Simulamos que un LLM (ej: ChatGPT + vision) extrae datos de una imagen
class DocumentoExtraido(BaseModel):
    """
    Modelo para datos extraídos de un documento (cédula, licencia, etc)
    Garantiza que el LLM devuelve SIEMPRE la estructura correcta.
    """
    tipo: str               # "cedula", "licencia", "pasaporte"
    numero: str             # número del documento
    nombres: str
    apellidos: str
    fecha_nacimiento: str

print("\n📄 Supongamos que el LLM extrae de una imagen:")
print("   Tipo: cedula")
print("   Número: 12345678")
print("   Nombres: Juan")
print("   Apellidos: Perez")
print("   Fecha de nacimiento: 1990-05-15")

# Datos que el LLM extraería del documento (como JSON)
datos_llm = {
    "tipo": "cedula",
    "numero": "12345678",
    "nombres": "Juan",
    "apellidos": "Perez",
    "fecha_nacimiento": "1990-05-15"
}

# Validar que los datos sean correctos
doc = DocumentoExtraido(**datos_llm)
print(f"\n✅ Documento VALIDADO y CONFIABLE:")
print(f"   {doc}")

print(f"\n🔍 Podemos acceder a los campos de forma segura:")
print(f"   - Tipo de documento: {doc.tipo}")
print(f"   - Número: {doc.numero}")
print(f"   - Titular: {doc.nombres} {doc.apellidos}")
print(f"   - Fecha de nacimiento: {doc.fecha_nacimiento}")

print(f"\n💡 Ventajas:")
print(f"   ✅ Si el LLM omite un campo → ERROR (detectado)")
print(f"   ✅ Si el LLM envía tipos incorrectos → ERROR (detectado)")
print(f"   ✅ Podemos usar los datos con seguridad en la aplicación")